<a href="https://colab.research.google.com/github/Gelato1337/LLM-Workshop/blob/main/llm_information_extraction_workshop_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Information Extraction Workshop

Three pipelines for turning unstructured text into structured data — all running locally via Ollama.

1. **Two-pass aspect extraction** — discover categories from data, then extract them consistently
2. **Theme extraction with grounded quotes** — summarize content and verify evidence
3. **GABRIEL attribute rating** — turn qualitative text into numeric scores (OpenAI's social-science toolkit, pointed at local Ollama)

We use the IMDB movie-reviews dataset throughout.

---
## Setup

Install Ollama, pull a model, install Python dependencies. Run once per Colab session.

In [ ]:
# Install Ollama and start the server in the background.
# Pipeline 3 needs Ollama's /v1/responses endpoint (added in v0.13.3).
# The install script pulls the latest stable release, so we're fine.

# zstd is needed to extract the Ollama tarball (Colab doesn't ship it)
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # wait for the server to come up

!ollama --version

In [ ]:
# Python dependencies.
# - datasets: HuggingFace IMDB loader
# - openai-gabriel: Pipeline 3 (talks to Ollama via the openai SDK)
# - openai: GABRIEL's underlying client
!pip install -q datasets pandas tqdm openai openai-gabriel

import json
import requests
import pandas as pd
from pprint import pprint
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

In [ ]:
# Quick GPU check so we know what model size is reasonable.
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — stick to the smallest model (gemma4:e2b)")

In [ ]:
#In case server crashes rerun this
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # wait for the server to come up

!ollama --version

### Pick a model

All three options run on 16GB VRAM with headroom for long contexts. Both Qwen 3.5 and Gemma 4 are brand-new (released weeks ago).

- **Recommended — `gemma4:e4b`** (9.6GB, 128K context). Google's newest, released April 2026.
- **Alternative — `qwen3.5:9b`** (6.6GB, 256K context). Strong instruction-following — important since every pipeline asks for JSON.

- **Small model for fallback — `gemma4:e2b`** (7.2GB, 128K context). If VRAM is tight.

Full catalog: https://ollama.com/library

In [ ]:
#MODEL_NAME = "qwen3.5:9b"
MODEL_NAME = "gemma4:e4b"
#MODEL_NAME = "gemma4:e2b"

!ollama pull {MODEL_NAME}
print(f"Model {MODEL_NAME} ready")

In [ ]:
# Core helper for Pipelines 1 and 2.
# One function, all LLM calls go through it. format="json" makes Ollama
# return valid JSON so we can parse it directly.
OLLAMA_URL = "http://localhost:11434/api/chat"

def chat(prompt, json_mode=False):
    """Send a prompt to the local Ollama model and return the reply as text."""
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }
    if json_mode:
        payload["format"] = "json"  # force JSON output
    response = requests.post(OLLAMA_URL, json=payload).json()
    return response["message"]["content"]

# Smoke test
print(chat("In one sentence: tell me joke about finland?"))

In [ ]:
# Load 1000 IMDB reviews. Same dataset for all three pipelines.
ds = load_dataset("imdb", split="train[:1000]")
df = pd.DataFrame({"id": [f"imdb_{i}" for i in range(len(ds))], "text": ds["text"]})

print(f"Loaded {len(df)} reviews")
df.head()

In [ ]:
# Glance at one review so we know what we're working with.
sample = df.iloc[2]
print(f"ID: {sample['id']}  |  Length: {len(sample['text'])} chars")
print("-" * 50)
print(sample["text"][:800], "...")

---
## Pipeline 1 — Two-Pass Aspect Extraction

**The problem:** asking an LLM to extract "aspects" from reviews gives inconsistent labels across documents — "acting", "performance", "the actors", "cast" all mean the same thing. Aggregation falls apart.

**The fix — two passes:**
1. **Pass 1 (discovery).** Let the model freely extract aspects on a small sample. Count which names appear most often.
2. **Pass 2 (structured).** Pick a clean vocabulary based on what we saw, then force the model to use only those labels.

In [ ]:
# Pass 1: free-form discovery. No fixed categories — we want to see
# what labels the model invents so we can consolidate them ourselves.
PASS1_PROMPT = """Extract aspects discussed in this movie review.

For each aspect return an object with:
- "aspect": short name (e.g. "acting", "plot", "music")
- "sentiment": "positive", "negative", or "neutral"
- "quote": short supporting quote from the review

Return JSON: {"aspects": [ ... ]}

REVIEW:
"""

def extract_aspects_freeform(text):
    reply = chat(PASS1_PROMPT + text[:2500], json_mode=True)
    return json.loads(reply).get("aspects", [])

# Try it on one review first
pprint(extract_aspects_freeform(df.iloc[2]["text"]))

In [ ]:
# Run Pass 1 on a batch and collect aspect names.
# Watch which labels come up — they tell us what vocabulary to use in Pass 2.
N_PASS1 = 20

all_aspects, pass1_rows = [], []
for _, row in tqdm(df.head(N_PASS1).iterrows(), total=N_PASS1):
    try:
        for asp in extract_aspects_freeform(row["text"]):
            pass1_rows.append({"id": row["id"], **asp})
            all_aspects.append(asp["aspect"].lower())
    except Exception as e:
        print(f"Skipped {row['id']}: {e}")

print(f"\nFound {len(pass1_rows)} aspects across {N_PASS1} reviews\n")
print("Most common aspect names:")
pprint(Counter(all_aspects).most_common(20))

In [ ]:
# Look at the counts above, then hand-pick a clean taxonomy.
# This is the human-in-the-loop step — you decide what the categories should be.
FINAL_CATEGORIES = [
    "Acting", "Plot", "Directing", "Cinematography",
    "Music", "Dialogue", "Pacing", "Emotional Impact",
    "Comedy", "Recommendation", "Movie quality",
]

In [ ]:
# Pass 2: same task, but the model is forced to use only our fixed categories.
# Output is now consistent across reviews and ready for aggregation.
categories_list = "\n".join(f"- {c}" for c in FINAL_CATEGORIES)

PASS2_PROMPT = f"""Extract aspects from a movie review.

You MUST use ONLY these categories (exact spelling):
{categories_list}

If something doesn't fit any category, use "Other".

For each aspect return:
- "category": one of the allowed categories
- "sentiment": "positive", "negative", or "neutral"
- "quote": short supporting quote

Return JSON: {{"aspects": [ ... ]}}

REVIEW:
"""

def extract_aspects_structured(text):
    reply = chat(PASS2_PROMPT + text[:2500], json_mode=True)
    return json.loads(reply).get("aspects", [])

# Test on one review
pprint(extract_aspects_structured(df.iloc[0]["text"]))

In [ ]:
# Run Pass 2 on a batch and aggregate.
N_PASS2 = 15

pass2_rows = []
for _, row in tqdm(df.head(N_PASS2).iterrows(), total=N_PASS2):
    try:
        for asp in extract_aspects_structured(row["text"]):
            pass2_rows.append({"doc_id": row["id"], **asp})
    except Exception as e:
        print(f"Skipped {row['id']}: {e}")

results_df = pd.DataFrame(pass2_rows)
print(f"\nExtracted {len(results_df)} aspects\n")
print("Category distribution:")
print(results_df["category"].value_counts())

results_df.to_csv("aspect_results.csv", index=False)
results_df.head(10)

---
## Pipeline 2 — Theme Extraction with Grounded Quotes

Extract higher-level themes from each review, with a summary and a quote copied from the source text. The quote is the evidence trail — it grounds the summary.

We then check which quotes actually appear in the source. Hallucinated quotes are a common LLM failure mode, and worth measuring.

In [ ]:
# Theme extractor — same pattern as Pipeline 1.
THEME_PROMPT = """Extract the main themes from this text.

For each theme return:
- "theme": short name
- "summary": 1-2 sentence summary of the theme in this text
- "quote": EXACT word-for-word quote from the text (copied, not paraphrased)
- "keywords": list of 3-5 relevant keywords

Return JSON: {"themes": [ ... ]}

TEXT:
"""

def extract_themes(text):
    reply = chat(THEME_PROMPT + text[:2500], json_mode=True)
    return json.loads(reply).get("themes", [])

pprint(extract_themes(df.iloc[50]["text"]))

In [ ]:
# Run on a batch.
N_THEMES = 15

theme_rows = []
for _, row in tqdm(df.head(N_THEMES).iterrows(), total=N_THEMES):
    try:
        for t in extract_themes(row["text"]):
            theme_rows.append({
                "chunk_id": row["id"],
                "source_text": row["text"],
                "theme": t.get("theme", ""),
                "summary": t.get("summary", ""),
                "quote": t.get("quote", ""),
                "keywords": ", ".join(t.get("keywords", [])),
            })
    except Exception as e:
        print(f"Skipped {row['id']}: {e}")

themes_df = pd.DataFrame(theme_rows)
print(f"\nFound {len(themes_df)} themes")
themes_df.head()

In [ ]:
# Quote grounding check.
# Simple exact-substring match — if the "quote" doesn't appear verbatim
# in the source text, it was paraphrased or hallucinated.
themes_df["quote_found"] = themes_df.apply(
    lambda r: r["quote"].lower().strip() in r["source_text"].lower(),
    axis=1,
)

n_ok = themes_df["quote_found"].sum()
print(f"{n_ok}/{len(themes_df)} quotes match the source exactly")
print(f"Modified or hallucinated: {len(themes_df) - n_ok}")

themes_df.to_csv("theme_results.csv", index=False)
themes_df[["chunk_id", "theme", "summary", "quote", "quote_found"]].head(10)

---
## Pipeline 3 — GABRIEL: Rating Attributes at Scale

Pipelines 1 and 2 *extract* things. GABRIEL *measures* things.

[**GABRIEL**](https://github.com/openai/GABRIEL) is OpenAI's open-source toolkit for social scientists. You describe an attribute in plain English ("how enthusiastic is this review?") and GABRIEL scores every document 0–100 on it. Qualitative data becomes a numeric DataFrame you can analyze with statistics.

**Paper:** *GPT as a Measurement Tool* (NBER WP 34834).

GABRIEL was built for the OpenAI API, but since Ollama v0.13.3 it also implements the `/v1/responses` endpoint that GABRIEL uses. We just point GABRIEL's client at local Ollama — no API key, no cost.

In [ ]:
import os

# Redirect the OpenAI client (which GABRIEL uses under the hood) to local Ollama.
# "ollama" is a placeholder — the SDK requires a non-empty key but Ollama ignores it.
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

import gabriel

In [ ]:
# Attributes are just a {name: plain-English description} dict.
# Add, remove, or rewrite these freely — GABRIEL handles whatever you give it.
attributes = {
    "enthusiasm":              "How enthusiastic or excited the reviewer sounds",
    "plot_focus":              "How much the review discusses the plot vs other aspects",
    "recommendation_strength": "How strongly the reviewer would recommend this film",
    "emotional_intensity":     "How emotionally charged the review is (positive or negative)",
}

# Small sample — GABRIEL parallelizes automatically, keep it short for the demo.
sample = df.head(20).copy()

# gabriel.rate is async. In Colab we can await directly at the top level.
ratings = await gabriel.rate(
    sample,
    column_name="text",
    attributes=attributes,
    save_dir="/content/gabriel_runs/imdb_ratings",
    model=MODEL_NAME,          # the local Ollama model from Setup
    n_runs=1,
    modality="text",
    reset_files=True,
)

ratings.head()

In [ ]:
# Now we have numbers — real quantitative data from qualitative text.
attribute_cols = list(attributes.keys())

print("Summary statistics:")
print(ratings[attribute_cols].describe().round(1))

print("\nCorrelation matrix:")
print(ratings[attribute_cols].corr().round(2))

In [ ]:
# Sanity check: enthusiasm and recommendation strength should be positively correlated.
import matplotlib.pyplot as plt

ratings.plot.scatter(x="enthusiasm", y="recommendation_strength", figsize=(6, 4))
plt.title("Enthusiasm vs. Recommendation Strength")
plt.grid(alpha=0.3)
plt.show()